In [3]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt
import pandas as pd
import geopandas as gpd

Data Assembler

In [6]:
import numpy as np
from rasterstats import zonal_stats
from shapely.ops import unary_union

# 1. Load Base Files
wv_counties = gpd.read_file("../data/processed/wv_counties.gpkg")
roads = gpd.read_file("../data/processed/wv_roads.gpkg").to_crs(wv_counties.crs)
fcc_data = pd.read_csv("../data/processed/fcc_wv_broadband.csv") # Path to your FCC file

# 2. Process FCC Data (Create the 'Target')
fcc_data['county_fips'] = fcc_data['block_geoid'].astype(str).str.zfill(15).str[:5]
fcc_data['is_served'] = fcc_data['max_advertised_download_speed'] >= 100
county_stats = fcc_data.groupby('county_fips')['is_served'].mean().reset_index()
county_stats.columns = ['county_fips', 'broadband_serve_pct']

# 3. Join FCC to Counties
wv_counties = wv_counties.merge(county_stats, left_on="GEOID", right_on="county_fips", how="inner")

# 4. Feature Engineering
# Population
stats = zonal_stats(wv_counties.to_crs(epsg=4326), "../data/raw/usa_pop_1km.tif", stats="sum")
wv_counties['total_pop'] = [s['sum'] if s['sum'] is not None else 0 for s in stats]

# Road Density
roads_in_county = gpd.sjoin(roads, wv_counties, predicate='within')
road_lengths = roads_in_county.groupby('NAME')['length'].sum()
wv_counties['road_density'] = wv_counties['NAME'].map(road_lengths).fillna(0)

# Dist to Backbone
backbone = roads[roads['highway'].isin(['motorway', 'trunk', 'primary'])]
wv_counties['dist_to_backbone_m'] = wv_counties.geometry.centroid.apply(lambda x: backbone.distance(x).min())

print("Data Assembly Complete. Features created:", wv_counties.columns.tolist())

/tmp/ipykernel_10794/3835390173.py:31: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  wv_counties['dist_to_backbone_m'] = wv_counties.geometry.centroid.apply(lambda x: backbone.distance(x).min())
/tmp/ipykernel_10794/3835390173.py:31: UserWarning: Geometry is in a geographic CRS. Results from 'distance' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  wv_counties['dist_to_backbone_m'] = wv_counties.geometry.centroid.apply(lambda x: backbone.distance(x).min())


Data Assembly Complete. Features created: ['STATEFP', 'COUNTYFP', 'COUNTYNS', 'GEOID', 'GEOIDFQ', 'NAME', 'NAMELSAD', 'LSAD', 'CLASSFP', 'MTFCC', 'CSAFP', 'CBSAFP', 'METDIVFP', 'FUNCSTAT', 'ALAND', 'AWATER', 'INTPTLAT', 'INTPTLON', 'geometry', 'county_fips', 'broadband_serve_pct', 'total_pop', 'road_density', 'dist_to_backbone_m']


In [4]:
# 1. Prepare the Data
# X = Features (The 'Why'), y = Target (The 'Result')
features = ['total_pop', 'road_density', 'dist_to_backbone_m']
wv_counties = gpd.read_file("../data/processed/wv_counties.gpkg").to_crs(epsg=4326)
X = wv_counties[features]
y = wv_counties['broadband_serve_pct']

# 2. Split data: 80% to train the brain, 20% to test it
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Initialize and Train the Random Forest
# n_estimators=100 means we are building 100 decision trees
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# 4. Make Predictions and Score
predictions = model.predict(X_test)
r2 = r2_score(y_test, predictions)

print(f"Model R-squared Score: {r2:.2f}")
print("Note: 1.0 is perfect prediction, 0.0 is guessing.")

KeyError: "None of [Index(['total_pop', 'road_density', 'dist_to_backbone_m'], dtype='str')] are in the [columns]"